# Fine-tune I-JEPA on locally rendered DotWall frames

Run top to bottom in a Colab GPU runtime. The first run renders **15 episodes** (one per layout), takes one real ViT-H/14 optimizer epoch, and checks the best checkpoint. Only after that succeeds will the later cells render the full manifest and train for 20 epochs.

Raw episodes, manifests copied for execution, and logs stay under `/content` and disappear when the runtime resets. The full run writes **only `jepa-best.pth.tar` and `jepa-latest.pth.tar`** to Drive. The released ImageNet checkpoint and the existing full episode manifest are read from Drive. If Colab resets, rerun the notebook: raw episodes must be rendered again, then training resumes from the Drive checkpoint. Budget roughly 19 GiB of local disk for the full raw frames, plus room for the released checkpoint and temporary files.

The full model may need a high-memory GPU. If the smoke run runs out of GPU memory, stop; the full run uses the same architecture.

## 1. Set up code, DINO-WM Wall, and GPU

This branch must include the notebook and `logging.checkpoint_folder` support before cloning it in Colab.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

REPO = Path("/content/i-jepa-wm")
DINO_ROOT = Path("/content/dino_wm")
DINO_COMMIT = "0a9492fa12044b852ae9e001cc74604b79c8bb0c"
BRANCH = "wall-layout-generalization"

def run(command):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], cwd=REPO, check=True)

if not REPO.exists():
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        "https://github.com/kmylonas/i-jepa-wm.git", str(REPO),
    ], check=True)
else:
    run(["git", "fetch", "origin", BRANCH])
    run(["git", "switch", BRANCH])
    run(["git", "pull", "--ff-only"])

if not DINO_ROOT.exists():
    run(["git", "clone", "https://github.com/gaoyuezhou/dino_wm.git", DINO_ROOT])
run(["git", "-C", DINO_ROOT, "checkout", DINO_COMMIT])
run([sys.executable, "-m", "pip", "install", "-q",
     "gym==0.26.2", "omegaconf", "scipy", "pyyaml"])

import torch
assert torch.cuda.is_available(), "Select a GPU runtime before continuing"
assert "checkpoint_folder" in (REPO / "src/train.py").read_text(), (
    "Colab has an older branch; push/pull the notebook's trainer changes first")
print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", torch.cuda.is_bf16_supported())
print("I-JEPA commit:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip())

## 2. Keep inputs and generated data in the right places

Edit `SOURCE_DRIVE` if your **full** released checkpoint is elsewhere. The existing Drive manifest is read only. A local copy of `environment.json` points to this runtime's DINO-WM checkout.

In [ ]:
import json
import shutil
import yaml

DRIVE_ROOT = Path("/content/drive/MyDrive/datasets/wall_layout_generalization")
SOURCE_DRIVE = Path("/content/drive/MyDrive/checkpoints/IN1K-vit.h.14-300e.pth.tar")
MANIFEST_DRIVE = DRIVE_ROOT / "episode_manifest.json"
ENV_DRIVE = DRIVE_ROOT / "environment.json"
LOCAL_ROOT = Path("/content/wall_ijepa_finetune")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_LOCAL = LOCAL_ROOT / "IN1K-vit.h.14-300e.pth.tar"
ENV_LOCAL = LOCAL_ROOT / "environment.json"
FULL_ROOT = LOCAL_ROOT / "full"
FULL_ROOT.mkdir(parents=True, exist_ok=True)
FULL_MANIFEST = FULL_ROOT / "episode_manifest.json"

for path in (SOURCE_DRIVE, MANIFEST_DRIVE, ENV_DRIVE,
             DINO_ROOT / "env/wall/envs/wall.py"):
    assert path.is_file(), f"Missing input: {path}"
if not SOURCE_LOCAL.is_file() or SOURCE_LOCAL.stat().st_size != SOURCE_DRIVE.stat().st_size:
    shutil.copy2(SOURCE_DRIVE, SOURCE_LOCAL)
shutil.copy2(MANIFEST_DRIVE, FULL_MANIFEST)
settings = json.loads(ENV_DRIVE.read_text())
settings["dino_wm_root"] = str(DINO_ROOT)
ENV_LOCAL.write_text(json.dumps(settings, indent=2))

sys.path.insert(0, str(REPO))
from src.mylonas_ijepa_wm.wall_layout_manifest import load_manifest, prepare_manifest
from src.datasets.wall_frames import build_frame_index
full_manifest = load_manifest(FULL_MANIFEST)
print("Manifest episodes:", len(full_manifest["episodes"]))
print("Local free GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))

## 3. Helpers for rendering and training

Training runs in its own Python process. A failed render or training process raises an error, so later cells cannot silently treat it as a successful smoke test.

In [ ]:
def render_episodes(manifest_path, raw_dir, start, end):
    raw_dir.mkdir(parents=True, exist_ok=True)
    run([
        sys.executable, "-u", "-m", "src.mylonas_ijepa_wm.generate_episodes",
        "--env", "wall", "--env-config", ENV_LOCAL,
        "--episode-manifest", manifest_path, "--output-dir", raw_dir,
        "--steps-per-episode", "50", "--start-index", str(start),
        "--end-index", str(end), "--resume",
    ])

def training_config(manifest_path, raw_dir, log_dir, checkpoint_dir,
                    *, epochs, frame_stride, accumulation_steps, warmup):
    config = yaml.safe_load((REPO / "configs/wall_ijepa_finetune.yaml").read_text())
    config["meta"]["init_checkpoint"] = str(SOURCE_LOCAL)
    config["meta"]["use_bfloat16"] = bool(torch.cuda.is_bf16_supported())
    config["data"]["raw_dir"] = str(raw_dir)
    config["data"]["episode_manifest"] = str(manifest_path)
    config["data"]["frame_stride"] = frame_stride
    config["data"]["num_workers"] = 0
    config["optimization"]["epochs"] = epochs
    config["optimization"]["warmup"] = warmup
    config["optimization"]["accumulation_steps"] = accumulation_steps
    config["logging"]["folder"] = str(log_dir)
    config["logging"]["checkpoint_folder"] = str(checkpoint_dir)
    return config

TRAIN_CODE = (
    "import sys, yaml\n"
    "from src.train import main\n"
    "with open(sys.argv[1]) as stream:\n"
    "    config = yaml.safe_load(stream)\n"
    "main(config)\n"
)

def train_from_config(config, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(config))
    run([sys.executable, "-u", "-c", TRAIN_CODE, path])

## 4. Smoke test: render 15 episodes and train ViT-H/14 for one epoch

The Wall manifest requires all 9 training layouts, 2 validation layouts, and 4 test layouts; one episode per layout is its smallest valid dataset. A frame stride of 50 gives 18 train and 4 validation frames. This tests the real renderer, full released checkpoint, masked prediction, optimizer, validation, and checkpoint export. It is **not** a useful fine-tuned model.

In [ ]:
import math
import time

SMOKE_ROOT = LOCAL_ROOT / "smoke"
SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
SMOKE_MANIFEST = SMOKE_ROOT / "episode_manifest.json"
SMOKE_RAW = SMOKE_ROOT / "raw"
SMOKE_MANIFEST.write_text(json.dumps(prepare_manifest(
    base_seed=100_000, train_episodes=9, val_episodes=2, test_episodes=4,
), indent=2))
render_episodes(SMOKE_MANIFEST, SMOKE_RAW, 0, 15)
smoke_index = build_frame_index(SMOKE_RAW, load_manifest(SMOKE_MANIFEST), frame_stride=50)
assert smoke_index["sample_counts"] == {"train": 18, "val": 4, "test": 8}
print("Smoke frame counts:", smoke_index["sample_counts"])

SMOKE_RUN = LOCAL_ROOT / "runs" / f"smoke_{int(time.time())}"
smoke_config = training_config(
    SMOKE_MANIFEST, SMOKE_RAW, SMOKE_RUN / "logs", SMOKE_RUN / "checkpoints",
    epochs=1, frame_stride=50, accumulation_steps=4, warmup=0,
)
train_from_config(smoke_config, SMOKE_RUN / "config.yaml")

smoke_best = SMOKE_RUN / "checkpoints/jepa-best.pth.tar"
smoke_latest = SMOKE_RUN / "checkpoints/jepa-latest.pth.tar"
assert smoke_best.is_file() and smoke_latest.is_file()
smoke = torch.load(smoke_best, map_location="cpu", weights_only=True, mmap=True)
assert smoke["epoch"] == 1 and smoke["optimizer_updates"] > 0
assert math.isfinite(smoke["val_loss"])
source = torch.load(SOURCE_LOCAL, map_location="cpu", weights_only=True, mmap=True)
weight_key = "patch_embed.proj.weight"
assert weight_key in smoke["encoder"]
source_key = weight_key if weight_key in source["encoder"] else "module." + weight_key
assert not torch.equal(smoke["encoder"][weight_key], source["encoder"][source_key]), (
    "The online encoder did not update during the smoke run")
del source, smoke
SMOKE_PASSED = True
print("SMOKE PASSED: one training epoch, finite validation loss, updated encoder")

## 5. Generate all DotWall episodes locally

This cell runs only after the smoke passed. The current dataset loader validates **all** manifest episodes, including the held-out test split, although training uses only train and validation frames. Nothing generated here is saved to Drive. Colab runtime reset means rerendering these episodes before resuming training.

In [ ]:
assert SMOKE_PASSED is True, "Run the smoke cell successfully first"
FULL_RAW = FULL_ROOT / "raw"
count = len(full_manifest["episodes"])
bytes_per_episode = (full_manifest["steps_per_episode"] + 1) * 224 * 224 * 3
missing = sum(not (FULL_RAW / f"episode_{i:04d}.pt").is_file()
              for i in range(count))
free = shutil.disk_usage("/content").free
needed = missing * bytes_per_episode + 5 * 2**30
print(f"Episodes still to render: {missing}/{count}")
print(f"Estimated raw bytes still needed: {missing * bytes_per_episode / 2**30:.1f} GiB")
print(f"Free local disk: {free / 2**30:.1f} GiB")
assert free > needed, "Insufficient /content disk; use a runtime with more local storage"

for start in range(0, count, 100):
    render_episodes(FULL_MANIFEST, FULL_RAW, start, min(start + 100, count))

full_index = build_frame_index(FULL_RAW, full_manifest, frame_stride=5)
print("Full frame counts:", full_index["sample_counts"])
FULL_DATA_READY = True

## 6. Fine-tune on the full dataset

Only the **best** and **latest** full training checkpoints go to Drive. `best` is replaced only when validation loss improves; `latest` supports resume after a Colab reset. Raw episodes and logs stay under `/content`. Keep the same checkpoint directory and configuration when resuming.

In [ ]:
assert SMOKE_PASSED is True and FULL_DATA_READY is True
FULL_LOGS = LOCAL_ROOT / "runs/full/logs"
FULL_CHECKPOINTS = DRIVE_ROOT / "ijepa_finetune_checkpoints_v1"
latest_path = FULL_CHECKPOINTS / "jepa-latest.pth.tar"
best_path = FULL_CHECKPOINTS / "jepa-best.pth.tar"
resume = latest_path.is_file() or best_path.is_file()
full_config = training_config(
    FULL_MANIFEST, FULL_RAW, FULL_LOGS, FULL_CHECKPOINTS,
    epochs=20, frame_stride=5, accumulation_steps=16, warmup=2,
)
full_config["meta"]["load_checkpoint"] = resume
print("Resume from Drive checkpoint:", resume)
print("Only checkpoint files will be written to:", FULL_CHECKPOINTS)
train_from_config(full_config, LOCAL_ROOT / "runs/full/config.yaml")

assert best_path.is_file(), "Training finished without a best checkpoint"
best = torch.load(best_path, map_location="cpu", weights_only=True, mmap=True)
assert math.isfinite(best["val_loss"])
print("Best epoch:", best["epoch"], "validation loss:", best["val_loss"])
print("Best full checkpoint:", best_path)

## 7. Optional: export the selected online encoder checkpoint

This saves one more **checkpoint file** to Drive. It is the format expected by the existing Wall embedding pipeline; no embeddings or raw episodes are written to Drive here.

In [ ]:
assert best_path.is_file(), "Run full fine-tuning first"
encoder_path = FULL_CHECKPOINTS / "jepa-best.encoder-only.pth.tar"
run([
    sys.executable, "-m", "src.mylonas_ijepa_wm.export_wall_ijepa_encoder",
    "--checkpoint", best_path, "--output", encoder_path,
])
assert encoder_path.is_file()
print("Fine-tuned online encoder:", encoder_path)